# LangChain Document Loaders - Complete Tutorial

Document loaders are the **first step** in any RAG pipeline. They read raw data from different sources and return a list of `Document` objects that LangChain can work with.

---

## What is a `Document`?

Every loader returns a list of `Document` objects. Each `Document` has two fields:

| Field | Type | Description |
|---|---|---|
| `page_content` | `str` | The actual text of the document |
| `metadata` | `dict` | Source info like file path, page number, etc. |

---

## Loaders Covered in This Tutorial

1. `TextLoader` — plain `.txt` files
2. `PyPDFLoader` — PDF files
3. `CSVLoader` — CSV/tabular files
4. `WebBaseLoader` — web pages from URLs
5. `DirectoryLoader` — entire folders at once
6. Bonus: Inspecting & filtering documents

## Setup — Install & Import Dependencies

In [ ]:
# Verify key packages are available
import importlib

packages = ["langchain", "langchain_community", "pypdf"]
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"✅ {pkg} is installed")
    except ImportError:
        print(f"❌ {pkg} is NOT installed — run: pip install {pkg}")

: 

In [2]:
import os
from pathlib import Path

# All sample data lives in the ../data/ folder
DATA_DIR = Path("../data")

print("Files available for loading:")
for f in DATA_DIR.iterdir():
    print(f"  {f.name}")

Files available for loading:
  employees.csv
  sample.txt
  notes.txt


---
## 1. TextLoader — Load Plain Text Files

`TextLoader` reads a `.txt` file and returns it as a **single Document**.
The entire file content goes into `page_content`.

In [3]:
from langchain_community.document_loaders import TextLoader

# Load a single .txt file
loader = TextLoader(DATA_DIR / "sample.txt")
docs = loader.load()

print(f"Number of documents loaded: {len(docs)}")
print(f"Type of each item       : {type(docs[0])}")

/home/abhinav/myProjects/Langchain-RAG-LLM-AI-Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of documents loaded: 1
Type of each item       : <class 'langchain_core.documents.base.Document'>


In [4]:
doc = docs[0]

print("=== page_content (first 300 chars) ===")
print(doc.page_content[:300])

print("\n=== metadata ===")
print(doc.metadata)

=== page_content (first 300 chars) ===
LangChain Document Loaders - Overview

LangChain provides a unified interface for loading documents from many different sources.
Document loaders are a key part of the RAG (Retrieval-Augmented Generation) pipeline.

What is a Document?
-------------------
In La

=== metadata ===
{'source': '../data/sample.txt'}


In [5]:
# TextLoader supports different encodings — useful for non-ASCII files
loader_utf8 = TextLoader(DATA_DIR / "sample.txt", encoding="utf-8")
docs_utf8 = loader_utf8.load()

print(f"Loaded {len(docs_utf8)} document(s) with UTF-8 encoding")
print(f"Character count: {len(docs_utf8[0].page_content)}")

Loaded 1 document(s) with UTF-8 encoding
Character count: 1143


---
## 2. PyPDFLoader — Load PDF Files

`PyPDFLoader` reads a PDF and returns **one Document per page**.  
Metadata includes the `source` path and `page` number (0-indexed).

> We'll create a small sample PDF using `reportlab` just for this demo.

In [9]:
# Create a sample PDF to load (uses only stdlib + reportlab)
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas as rl_canvas

    pdf_path = DATA_DIR / "sample.pdf"
    c = rl_canvas.Canvas(str(pdf_path), pagesize=letter)

    # Page 1
    c.setFont("Helvetica-Bold", 16)
    c.drawString(72, 720, "LangChain RAG Pipeline - Page 1")
    c.setFont("Helvetica", 12)
    c.drawString(72, 680, "RAG stands for Retrieval-Augmented Generation.")
    c.drawString(72, 660, "It combines a retrieval system with a language model.")
    c.drawString(72, 640, "Step 1: Load documents using Document Loaders.")
    c.drawString(72, 620, "Step 2: Split documents into chunks.")
    c.showPage()

    # Page 2
    c.setFont("Helvetica-Bold", 16)
    c.drawString(72, 720, "LangChain RAG Pipeline - Page 2")
    c.setFont("Helvetica", 12)
    c.drawString(72, 680, "Step 3: Create embeddings from the chunks.")
    c.drawString(72, 660, "Step 4: Store embeddings in a vector database.")
    c.drawString(72, 640, "Step 5: Retrieve relevant chunks for a query.")
    c.drawString(72, 620, "Step 6: Pass retrieved context to the LLM.")
    c.showPage()

    c.save()
    print(f"✅ Sample PDF created at: {pdf_path}")

except ImportError:
    print("reportlab not installed. Install it with: pip install reportlab")
    print("Alternatively, place any existing PDF at data/sample.pdf")

✅ Sample PDF created at: ../data/sample.pdf


In [10]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = DATA_DIR / "sample.pdf"

loader = PyPDFLoader(str(pdf_path))
pdf_docs = loader.load()

print(f"Pages loaded: {len(pdf_docs)}")
print()

for i, doc in enumerate(pdf_docs):
    print(f"--- Page {i+1} ---")
    print(f"Content : {doc.page_content.strip()[:120]}...")
    print(f"Metadata: {doc.metadata}")
    print()

Pages loaded: 2

--- Page 1 ---
Content : LangChain RAG Pipeline - Page 1
RAG stands for Retrieval-Augmented Generation.
It combines a retrieval system with a lan...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-09T08:22:06+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-03-09T08:22:06+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': '../data/sample.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}

--- Page 2 ---
Content : LangChain RAG Pipeline - Page 2
Step 3: Create embeddings from the chunks.
Step 4: Store embeddings in a vector database...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-09T08:22:06+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-03-09T08:22:06+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': '../data/sample.pdf', 'total_pages':

In [ ]:
# You can also use lazy_load() to avoid loading everything into memory at once
# Great for large PDFs!
loader = PyPDFLoader(str(pdf_path))

print("Lazily iterating pages:")
for page in loader.lazy_load():
    print(f"  Page {page.metadata['page']}: {len(page.page_content)} chars")

---
## 3. CSVLoader — Load CSV Files

`CSVLoader` reads a CSV and returns **one Document per row**.  
Each document's `page_content` is a formatted key: value string for that row.

In [11]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path=str(DATA_DIR / "employees.csv"))
csv_docs = loader.load()

print(f"Rows loaded as documents: {len(csv_docs)}")

ModuleNotFoundError: No module named 'langchain_community.document_loaders.csv_loader'

In [ ]:
# Each row becomes its own Document
for doc in csv_docs[:3]:
    print("--- Document ---")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}")
    print()

In [ ]:
# Use a specific column as the source identifier in metadata
loader_with_source = CSVLoader(
    file_path=str(DATA_DIR / "employees.csv"),
    source_column="name"          # The 'name' column becomes the metadata source
)
csv_docs_named = loader_with_source.load()

print("Metadata with custom source column:")
for doc in csv_docs_named[:2]:
    print(f"  source: {doc.metadata['source']}, row: {doc.metadata['row']}")

---
## 4. WebBaseLoader — Load Web Pages

`WebBaseLoader` fetches a URL and extracts its text content using `BeautifulSoup`.  
You can pass multiple URLs and it loads them all.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# Load a public webpage (LangChain docs page as example)
url = "https://python.langchain.com/docs/introduction/"

loader = WebBaseLoader(url)
web_docs = loader.load()

print(f"Documents loaded: {len(web_docs)}")
print(f"\nMetadata: {web_docs[0].metadata}")
print(f"\nContent preview (first 500 chars):")
print(web_docs[0].page_content[:500])

In [ ]:
# Load MULTIPLE URLs at once — returns one Document per URL
urls = [
    "https://python.langchain.com/docs/introduction/",
    "https://python.langchain.com/docs/concepts/",
]

multi_loader = WebBaseLoader(urls)
multi_docs = multi_loader.load()

print(f"Total documents from {len(urls)} URLs: {len(multi_docs)}")
for doc in multi_docs:
    print(f"  Source: {doc.metadata.get('source', 'N/A')} | Chars: {len(doc.page_content)}")

---
## 5. DirectoryLoader — Load All Files in a Folder

`DirectoryLoader` scans a directory and loads all matching files using a specified sub-loader.  
Use the `glob` parameter to filter by file extension.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

# Load all .txt files from the data directory
loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.txt",             # Matches all .txt files recursively
    loader_cls=TextLoader,       # Which loader to use for each file
    show_progress=True,          # Show a progress bar
    use_multithreading=True,     # Load files in parallel (faster for many files)
)

dir_docs = loader.load()
print(f"\nTotal documents loaded: {len(dir_docs)}")

In [ ]:
for doc in dir_docs:
    print(f"Source: {doc.metadata['source']}")
    print(f"Length: {len(doc.page_content)} chars")
    print(f"Preview: {doc.page_content[:80].strip()}...")
    print()

In [ ]:
# Load CSV files from the same directory
from langchain_community.document_loaders.csv_loader import CSVLoader

csv_dir_loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.csv",
    loader_cls=CSVLoader,
)

csv_dir_docs = csv_dir_loader.load()
print(f"CSV documents loaded: {len(csv_dir_docs)}")

---
## 6. Working with Documents — Inspect, Filter & Combine

Once loaded, documents are just Python objects. You can filter, inspect, and manipulate them freely.

In [ ]:
# Combine documents from multiple loaders
all_docs = dir_docs + csv_dir_docs
print(f"Total combined documents: {len(all_docs)}")

In [ ]:
# Inspect document lengths
for doc in all_docs:
    source = doc.metadata.get("source", "unknown")
    row = doc.metadata.get("row", "-")
    print(f"{Path(source).name:<20} | row={row:<3} | {len(doc.page_content)} chars")

In [ ]:
# Filter: only keep documents longer than 200 characters
long_docs = [doc for doc in all_docs if len(doc.page_content) > 200]
print(f"Documents with > 200 chars: {len(long_docs)} out of {len(all_docs)}")

In [ ]:
# Add custom metadata to documents — useful before storing in a vector DB
for doc in all_docs:
    doc.metadata["project"] = "langchain-rag-tutorial"
    doc.metadata["loaded_by"] = "document_loaders_tutorial.ipynb"

print("Updated metadata on all documents:")
print(all_docs[0].metadata)

---
## 7. What Comes Next — The Full RAG Pipeline

Loading documents is just Step 1. Here's the full pipeline and where document loaders fit:

In [ ]:
pipeline_steps = [
    ("1", "Document Loading",   "TextLoader, PyPDFLoader, CSVLoader, WebBaseLoader  ← YOU ARE HERE"),
    ("2", "Text Splitting",     "RecursiveCharacterTextSplitter, TokenTextSplitter"),
    ("3", "Embedding Creation", "OpenAIEmbeddings, HuggingFaceEmbeddings"),
    ("4", "Vector Storage",     "FAISS, ChromaDB, Pinecone"),
    ("5", "Retrieval",          "VectorStoreRetriever, MultiQueryRetriever"),
    ("6", "Generation",         "ChatOpenAI, ChatGroq + RetrievalQA chain"),
]

print(f"{'Step':<6} {'Stage':<22} {'Tools / Components'}")
print("-" * 80)
for step, stage, tools in pipeline_steps:
    print(f"  {step}    {stage:<22} {tools}")

---
## Summary

| Loader | Best For | Output |
|---|---|---|
| `TextLoader` | Plain `.txt` files | 1 doc per file |
| `PyPDFLoader` | PDF files | 1 doc per page |
| `CSVLoader` | Tabular/CSV data | 1 doc per row |
| `WebBaseLoader` | Web URLs | 1 doc per URL |
| `DirectoryLoader` | Entire folders | Delegates to sub-loader |

**Key Takeaways:**
- All loaders return `List[Document]` with `page_content` and `metadata`
- Use `lazy_load()` for memory-efficient loading of large files
- You can add custom `metadata` to any document before storing
- `DirectoryLoader` is great for bulk loading with `show_progress=True`

**Next Tutorial:** Text Splitters — how to chunk documents for embedding